# Tutorial: Run FIPpy on a HYDRAD Output Folder

This notebook shows the minimal public workflow for using this repository with your own HYDRAD output. It reads one HYDRAD strand, runs the Alfven-wave and FIP-bias post-processing pipeline, and recreates representative paper-style plots from the returned `output_data` dictionary.

The full paper workflow notebook, `main_chianti.ipynb`, is intentionally not included because it contains local paths and generated paper products.

## 1. Setup

Install the dependencies from `requirements.txt`, then edit the parameters below. `HYDRAD_DIR` must point to a HYDRAD output folder that `pydrad` can parse.

The processing function writes a derived pickle and diagnostic figures under the repository `results/` directory. Those outputs are ignored by Git.

In [ ]:
from pathlib import Path
import sys

import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np

REPO_ROOT = Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from loop.reader import HydradReader
from main_fip_bias_batch import process_hydrad_data

plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.grid": True,
    "grid.alpha": 0.25,
})

In [ ]:
# Edit these values for your HYDRAD run.
HYDRAD_DIR = Path("/path/to/HYDRAD_output")
STRAND_NUM = 700

FREQUENCY = 0.206 * u.s**-1
AMPLITUDE = 0.97 * u.km / u.s
ACOUSTIC_FLUX = 1e8 * u.erg / u.cm**2 / u.s
V_ADD = 0.0 * u.km / u.s

if not HYDRAD_DIR.exists():
    raise FileNotFoundError(
        f"Set HYDRAD_DIR to an existing HYDRAD output folder before running the tutorial: {HYDRAD_DIR}"
    )

## 2. Inspect One HYDRAD Strand

`HydradReader` is a thin wrapper around `pydrad.parse.parse.Strand`. This quick check confirms that the folder and strand number are valid before running the full FIPpy pipeline.

In [ ]:
hydrad_strand = HydradReader(str(HYDRAD_DIR), strand_num=STRAND_NUM).read_strand()
print(type(hydrad_strand))

for name in ["time", "coordinate", "electron_density", "electron_temperature"]:
    if hasattr(hydrad_strand, name):
        value = getattr(hydrad_strand, name)
        try:
            print(f"{name}: shape={np.shape(value)}")
        except Exception:
            print(f"{name}: {type(value)}")

## 3. Run The FIPpy Post-Processing Pipeline

This cell solves the wave equations and calculates the FIP-bias quantities for the selected strand. It may take several minutes depending on the HYDRAD output size and machine.

The second path-like argument is kept for compatibility with the original research driver; the current implementation derives the actual results directory internally.

In [ ]:
output_data = process_hydrad_data(
    input_file=str(HYDRAD_DIR),
    frequency=FREQUENCY,
    amplitude=AMPLITUDE,
    output_dir=str(HYDRAD_DIR),
    strand_num=STRAND_NUM,
    acoustic_flux=ACOUSTIC_FLUX,
    v_add=V_ADD,
)

print("Available output_data keys:")
for key in sorted(output_data):
    print(f"- {key}")

## 4. Plot Helpers

The functions below use only `output_data`, so you can rerun the plotting cells later from a saved result dictionary if needed.

In [ ]:
def quantity_values(values, unit=None):
    """Return a NumPy array, converting Astropy quantities when present."""
    if hasattr(values, "to") and unit is not None:
        return np.asarray(values.to(unit).value)
    if hasattr(values, "value"):
        return np.asarray(values.value)
    return np.asarray(values)


def loop_distance_mm(data):
    return quantity_values(data["s_values"], u.Mm)


def plot_loop_atmosphere(data):
    s_mm = loop_distance_mm(data)
    temperature = quantity_values(data["T"], u.K)
    density = quantity_values(data["ne"], u.cm**-3)

    fig, ax_density = plt.subplots(figsize=(8, 4.8))
    ax_temperature = ax_density.twinx()

    ax_density.semilogy(s_mm, density, color="tab:blue", label="Electron density")
    ax_temperature.semilogy(s_mm, temperature, color="tab:red", label="Temperature")

    ax_density.set_xlabel("Distance along loop (Mm)")
    ax_density.set_ylabel("Electron density (cm$^{-3}$)", color="tab:blue")
    ax_temperature.set_ylabel("Temperature (K)", color="tab:red")
    ax_density.tick_params(axis="y", labelcolor="tab:blue")
    ax_temperature.tick_params(axis="y", labelcolor="tab:red")
    ax_density.set_title("HYDRAD atmosphere used by FIPpy")

    lines, labels = ax_density.get_legend_handles_labels()
    lines2, labels2 = ax_temperature.get_legend_handles_labels()
    ax_density.legend(lines + lines2, labels + labels2, loc="best")
    fig.tight_layout()
    return fig


def plot_ponderomotive_acceleration(data):
    s_mm = loop_distance_mm(data)
    acceleration = quantity_values(data["ponderomotive_acceleration"], u.cm / u.s**2)

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(s_mm, acceleration, color="black")
    ax.axhline(0.0, color="0.5", linewidth=0.8)
    ax.set_xlabel("Distance along loop (Mm)")
    ax.set_ylabel("Ponderomotive acceleration (cm s$^{-2}$)")
    ax.set_title("Ponderomotive acceleration along the loop")
    fig.tight_layout()
    return fig


def plot_ionisation_fractions(data, elements=("Ca", "Fe", "Mg", "Si", "O", "Ne", "Ar")):
    s_mm = loop_distance_mm(data)
    fractions = data["ionization_fractions"]

    fig, ax = plt.subplots(figsize=(8, 4.8))
    for element in elements:
        if element in fractions:
            ax.semilogy(s_mm, np.clip(fractions[element], 1e-6, 1.0), label=element)

    ax.set_xlabel("Distance along loop (Mm)")
    ax.set_ylabel("Ionisation fraction")
    ax.set_ylim(1e-4, 1.1)
    ax.set_title("Ionisation fractions used in the fractionation calculation")
    ax.legend(ncol=2)
    fig.tight_layout()
    return fig


def bias_arrays(data, element, side):
    segment = data["fip_bias_heights"].get(element, {}).get(side)
    if segment is None:
        return None, None
    heights = quantity_values(segment["heights"])
    bias = quantity_values(segment["bias"])
    return heights, bias


def plot_fip_bias_relative_to_oxygen(data, side="right", elements=("Ca", "Fe", "Mg", "Si", "S", "Ar", "Ne")):
    oxygen_height, oxygen_bias = bias_arrays(data, "O", side)
    if oxygen_height is None:
        raise KeyError(f"No oxygen FIP-bias data found for side={side!r}")

    fig, ax = plt.subplots(figsize=(8, 4.8))
    for element in elements:
        heights, bias = bias_arrays(data, element, side)
        if heights is None:
            continue
        oxygen_at_height = np.interp(heights, oxygen_height, oxygen_bias)
        ax.plot(heights, bias / oxygen_at_height, label=f"{element}/O")

    ax.axhline(1.0, color="0.5", linewidth=0.8)
    ax.set_xlabel("Height (Mm)")
    ax.set_ylabel("FIP bias normalized to O")
    ax.set_title(f"FIP bias at the {side} footpoint")
    ax.legend(ncol=2)
    fig.tight_layout()
    return fig

## 5. Recreate Representative Paper-Style Plots

These are simplified versions of plots used in the paper workflow: loop atmosphere, ponderomotive acceleration, ionisation fractions, and FIP bias normalized to oxygen.

In [ ]:
plot_loop_atmosphere(output_data)
plot_ponderomotive_acceleration(output_data)
plot_ionisation_fractions(output_data)
plot_fip_bias_relative_to_oxygen(output_data, side="right")
plt.show()

## 6. Optional: Small Parameter Sweep

The paper explores how acoustic flux and additional turbulent velocity affect the predicted FIP bias. The example below runs a small `v_add` sweep. It can be expensive, so reduce or expand the values depending on your machine.

In [ ]:
RUN_V_ADD_SWEEP = False
V_ADD_SWEEP = [0.0, 5.0, 10.0] * u.km / u.s

sweep_results = []
if RUN_V_ADD_SWEEP:
    for v_add in V_ADD_SWEEP:
        result = process_hydrad_data(
            input_file=str(HYDRAD_DIR),
            frequency=FREQUENCY,
            amplitude=AMPLITUDE,
            output_dir=str(HYDRAD_DIR),
            strand_num=STRAND_NUM,
            acoustic_flux=ACOUSTIC_FLUX,
            v_add=v_add,
        )
        sweep_results.append((v_add, result))

    fig, ax = plt.subplots(figsize=(8, 4.8))
    for v_add, result in sweep_results:
        oxygen_height, oxygen_bias = bias_arrays(result, "O", "right")
        calcium_height, calcium_bias = bias_arrays(result, "Ca", "right")
        calcium_over_oxygen = calcium_bias / np.interp(calcium_height, oxygen_height, oxygen_bias)
        ax.plot(calcium_height, calcium_over_oxygen, label=f"v_add={v_add.to_value(u.km/u.s):.1f} km/s")

    ax.set_xlabel("Height (Mm)")
    ax.set_ylabel("Ca/O FIP bias")
    ax.set_title("Effect of added turbulent velocity")
    ax.legend()
    fig.tight_layout()
    plt.show()